In [1]:
# Jupyter cell: Initialize and optimize SQLite in FLY

# 1) Imports
from sqlalchemy import create_engine, text
from sqlalchemy.orm import sessionmaker

from infrastructure.config import Config



In [2]:
# 2) Load application configuration
#    Adjust constructor as needed (e.g., passing path to config file or env vars)
config = Config()



In [3]:
# 3) Create the SQLAlchemy engine with thread-safe and future settings
engine = create_engine(
    config.database.connection_string,
    connect_args={"check_same_thread": False},  # allow multi-thread access
    future=True,
)



In [ ]:
# 4) Execute SQLite PRAGMAs for performance & integrity
with engine.connect() as conn:
    conn.execute(text("PRAGMA wal_checkpoint"))

    # ou, se preferir garantir que todo o WAL seja aplicado e truncado:
    conn.execute(text("PRAGMA wal_checkpoint(FULL)"))
    conn.execute(text("PRAGMA wal_checkpoint(RESTART)"))
    conn.execute(text("PRAGMA optimize"))              # Run internal optimizations
    conn.execute(text("PRAGMA journal_mode=WAL"))      # Write-Ahead Logging
    conn.execute(text("PRAGMA synchronous=FULL"))      # Full disk sync for safety
    conn.execute(text("PRAGMA foreign_keys=ON"))       # Enforce FK constraints
    conn.execute(text("PRAGMA temp_store=MEMORY"))     # RAM for temp tables
    conn.execute(text("PRAGMA cache_size=-65536"))     # 64 MB cache



In [ ]:
# 5) Create a session factory for ORM transactions
Session = sessionmaker(
    bind=engine,
    autoflush=True,
    expire_on_commit=True,
)



In [ ]:
# 6) (Optional) Inspect existing tables
with Session() as session:
    result = session.execute(text("SELECT name FROM sqlite_master WHERE type='table';"))
    tables = [row[0] for row in result]
    print("⏺️ Tables in database:", tables)
